In [5]:
import torch
import torchaudio
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
import pandas as pd
import os
from torchvision import models

# 1. Đọc file CSV đã xử lý ở bước trước
df = pd.read_csv('Cleaned_Insect_Data.csv')

# 2. Mã hóa tên loài thành số (0-65)
le = LabelEncoder()
df['label'] = le.fit_transform(df['species'])
num_classes = len(le.classes_)

print(f"Tổng số loài: {num_classes}")
print(f"Ví dụ nhãn 0 là loài: {le.inverse_transform([0])[0]}")


Tổng số loài: 66
Ví dụ nhãn 0 là loài: Achetadomesticus


In [6]:
class InsectDataset(Dataset):
    def __init__(self, dataframe, audio_dir):
        self.df = dataframe
        self.audio_dir = audio_dir
        self.mel_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=22050, n_mels=128, n_fft=1024, hop_length=512
        )
        self.db_transform = torchaudio.transforms.AmplitudeToDB()

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        file_path = os.path.join(self.audio_dir, self.df.iloc[idx]['file_name'])
        label = self.df.iloc[idx]['label']
        
        # Load audio
        waveform, sr = torchaudio.load(file_path)
        
        # Chuyển sang Spectrogram
        spec = self.mel_transform(waveform)
        spec = self.db_transform(spec)
        
        return spec, label

# Chia tập Train và Validation
train_data = df[df['subset'] == 'train']
val_data = df[df['subset'] == 'validation']

train_loader = DataLoader(InsectDataset(train_data, 'Processed_Audio_5s'), batch_size=32, shuffle=True)
val_loader = DataLoader(InsectDataset(val_data, 'Processed_Audio_5s'), batch_size=32)

print(f"Đã chuẩn bị: {len(train_data)} mẫu train, {len(val_data)} mẫu validation.")


Đã chuẩn bị: 12009 mẫu train, 3692 mẫu validation.


In [7]:
def build_model(num_classes):
    # Dùng ResNet18 - Cân bằng giữa tốc độ và độ chính xác
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    
    # Thay đổi lớp đầu tiên để nhận Spectrogram 1 kênh (đen trắng)
    model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
    
    # Thay đổi lớp cuối cùng để phân loại đúng số lượng loài của bạn
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = build_model(num_classes).to(device)

print(f"Đang chạy trên thiết bị: {device}")


Đang chạy trên thiết bị: cuda


In [8]:
# --- COPY VÀ DÁN ĐÈ TOÀN BỘ VÀO CELL BỊ LỖI ---
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 10 

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for inputs, labels in train_loader:
        # CHÚ Ý: Chuyển inputs lên GPU và ép kiểu labels sang LONG để hết lỗi
        inputs = inputs.to(device)
        labels = labels.to(device).long() # << ĐÂY LÀ DÒNG QUAN TRỌNG NHẤT
        
        optimizer.zero_grad()
        outputs = model(inputs)
        
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
    print(f"Epoch [{epoch+1}/{epochs}] - Loss: {running_loss/len(train_loader):.4f} - Acc: {100.*correct/total:.2f}%")

# Lưu mô hình
torch.save(model.state_dict(), 'insect_detector.pth')
print("Đã lưu mô hình thành công!")


Epoch [1/10] - Loss: 1.2022 - Acc: 68.12%
Epoch [2/10] - Loss: 0.4555 - Acc: 86.55%
Epoch [3/10] - Loss: 0.2844 - Acc: 91.70%
Epoch [4/10] - Loss: 0.2174 - Acc: 93.65%
Epoch [5/10] - Loss: 0.1961 - Acc: 94.06%
Epoch [6/10] - Loss: 0.1277 - Acc: 96.39%
Epoch [7/10] - Loss: 0.1445 - Acc: 95.63%
Epoch [8/10] - Loss: 0.1193 - Acc: 96.62%
Epoch [9/10] - Loss: 0.1192 - Acc: 96.54%
Epoch [10/10] - Loss: 0.0712 - Acc: 97.92%
Đã lưu mô hình thành công!
